In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> price_line('GA', 89.5)\n'GA\\t$89.50'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> price_line('VIP', 249.0)\n'VIP\\t$249.00'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> price_line('Fees', 12.345)\n'Fees\\t$12.35'",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> order_code(7)\n'CS007'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> order_code(42)\n'CS042'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> order_code(1234)\n'CS1234'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> order_code(0)\n'CS000'",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> q3_notice == \'He said "sold out".\\nTry again '
                                       "tomorrow.'\n"
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q3_path == 'C:\\\\tickets\\\\new' and '\\t' not in "
                                       'q3_path\n'
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> totals([89.5, 249.0, 12.0])\n'
                                       '(3, 12.0, 249.0, 350.5)',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> totals([5.0])\n(1, 5.0, 5.0, 5.0)',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> totals([]) == (0, 0, 0, 0)\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q5_option.strip().upper() == 'C'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> to_cents('12.50')\n1250",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> to_cents('89.5')\n8950",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> to_cents('8.2')\n820",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> to_cents('0.29')\n29",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> to_cents('0')\n0",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q7': {'name': 'q7',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q7_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('05-Input-Output', _TESTS)
check = grader.check

print("Ready. 05-Input-Output loaded: 7 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Input/Output — Ticket Receipt

Tickets went on sale at 10:00. You got through. Now the site wants to show you what you
just bought, and there is a surprising amount of craft in making four lines of text look
right:

```
Ticket       Price
-------------------
GA           $89.50
VIP         $249.00
Fees         $12.35
```

Prices to exactly two decimal places, whether or not the number has them. Columns that
line up. An order number padded to a fixed width so nothing jumps around. This homework is
about the tools Python gives you for that — **f-strings**, **format specifiers** and
**escape sequences** — plus the other half of I/O: getting data *in* and converting it to
something you can compute with.

## Question 1 — One line of the receipt

Write `price_line(name, price)`, which returns one line of the receipt: the ticket name, a
**tab**, a dollar sign, and the price to exactly **two decimal places**.

```
price_line('GA', 89.5)      ->  'GA\t$89.50'
price_line('VIP', 249.0)    ->  'VIP\t$249.00'
price_line('Fees', 12.345)  ->  'Fees\t$12.35'
```

`\t` is the **tab** escape sequence — one character that tells the terminal to jump to the
next column stop. That is how the prices line up without you counting spaces.

An **f-string** is a string with an `f` in front of it, in which anything inside `{}` is
worked out and dropped in:

```python
name = "GA"
f"Ticket: {name}"        # 'Ticket: GA'
```

After a colon inside the braces you can say **how** to present the value. `:.2f` means
"as a fixed-point number with 2 digits after the point":

```python
f"{89.5:.2f}"     # '89.50'   -- pads out to two places
f"{12.345:.2f}"   # '12.35'   -- and rounds when there are too many
```

Note `price_line` **returns** the line. It does not print it.

In [ ]:
def price_line(name, price):
    ...

# try it out -- print() turns the \t into a real tab
print(price_line('GA', 89.5))
print(price_line('VIP', 249.0))

In [ ]:
grader.check("q1")

## Question 2 — Zero-padded order codes

Order numbers are shown as `CS` followed by the number padded with leading zeros to **at
least three digits**. A number that is already longer than three digits is left alone.

```
order_code(7)     ->  'CS007'
order_code(42)    ->  'CS042'
order_code(1234)  ->  'CS1234'
order_code(0)     ->  'CS000'
```

The specifier you want is `:03d` — "as an integer, at least 3 wide, padded with zeros".

```python
f"{7:03d}"     # '007'
f"{1234:03d}"  # '1234'  -- already wider than 3, so nothing is added
```

Padding like this is not decoration. Fixed-width codes sort correctly as text, line up in
a column, and can be found by a search that knows how long they are.

In [ ]:
def order_code(n):
    ...

# try it out
print(order_code(7), order_code(1234))

In [ ]:
grader.check("q2")

## Question 3 — Escape sequences

Some characters cannot simply be typed into a string. A newline would end the line. A
quote would end the string. So Python spells them with a backslash.

| Escape | Character |
|---|---|
| `\n` | a new line |
| `\t` | a tab |
| `\"` | a double quote |
| `\\` | a single backslash |

Build two strings:

**`q3_notice`** must be exactly these two lines — note the real double quotes around
*sold out*:

```
He said "sold out".
Try again tomorrow.
```

**`q3_path`** must be exactly this text, which is a Windows path:

```
C:\tickets\new
```

The second one is the trap, and it is worth meeting on a homework rather than at 2am. If
you type `"C:\tickets\new"` with single backslashes, Python reads `\t` as a **tab** and
`\n` as a **newline**, and your path silently becomes something else entirely. The test
checks specifically that there is no tab character in it.

Each string is worth 1 point.

In [ ]:
q3_notice = ...
q3_path = ...

print(q3_notice)
print(q3_path)

In [ ]:
grader.check("q3")

## Question 4 — The totals line

Write `totals(prices)`, which takes a list of prices and returns a tuple:

```
(how many, the cheapest, the dearest, the sum)
```

```
totals([89.5, 249.0, 12.0])  ->  (3, 12.0, 249.0, 350.5)
totals([5.0])                ->  (1, 5.0, 5.0, 5.0)
totals([])                   ->  (0, 0, 0, 0)
```

No loop needed — Python's built-ins do all four: `len()`, `min()`, `max()` and `sum()`.

Built-ins are **expressions**: each one evaluates to a value, so you can use a call
wherever a value would go, including inside another call. `max(len(a), len(b))` is a
perfectly ordinary thing to write, and Python works from the inside out — the `len()`
calls happen first, and `max()` receives their results.

The empty list needs handling before you get to `min()`, which raises a `ValueError` when
there is nothing to compare.

In [ ]:
def totals(prices):
    ...

# try it out
print(totals([89.5, 249.0, 12.0]))

In [ ]:
grader.check("q4")

## Question 5 — What `input()` hands you

The site asks how many tickets you want, and you type `3`:

```python
how_many = input("How many tickets? ")
```

What is in `how_many`?

**A** — the integer `3`

**B** — the integer `3` if you typed digits, and a string otherwise

**C** — the string `'3'`, always

Set `q5_option` to `'A'`, `'B'` or `'C'`.

This is the single most common source of confusion in first year, and it shows up as
`TypeError: can only concatenate str (not "int") to str`, or — worse — as
`"3" * 2` quietly giving you `'33'` instead of `6`. `input()` returns text. If you want a
number, you convert it, every time.

The cell below has a live `input()` in it, commented out. Uncomment it and run it to see
for yourself — but **comment it out again before you submit**, because a notebook that
stops to ask a question cannot be marked without somebody sitting there to answer it.

In [ ]:
# Uncomment these two lines to try it live, then comment them out again.
# how_many = input("How many tickets? ")
# print(how_many, type(how_many))

q5_option = ...

In [ ]:
grader.check("q5")

## Question 6 — Prices as integer cents

Since `input()` gives you text, a price arrives as something like `'12.50'`. Money is
safest handled as a whole number of **cents**, so write `to_cents(text)`, which converts a
price written as text into an `int` number of cents.

```
to_cents('12.50')  ->  1250
to_cents('89.5')   ->  8950
to_cents('8.2')    ->  820
to_cents('0.29')   ->  29
to_cents('0')      ->  0
```

Three steps, and they nest into one line once you see them:

1. `float(text)` turns the text into a number
2. multiply by 100
3. `round(...)` turns that back into a whole number

Step 3 is not optional, and it has to be `round()` rather than `int()`. `'8.2'` is why:
`float('8.2') * 100` is not 820, it is `819.9999999999999`, because 8.2 has no exact
binary representation. `int()` does not round — it **chops** — so it would hand you 819,
and you have just lost a cent of somebody's money. `round()` gives 820.

Two of the test cases are there specifically to catch `int()`. Try them both ways and
watch it happen.

In [ ]:
def to_cents(text):
    ...

# try it out -- and look at what the middle step actually produces
print(float('8.2') * 100)
print(to_cents('8.2'))

In [ ]:
grader.check("q6")

## Question 7 — `print()` versus `+`

Which of these prints exactly `12`?

**A** — `print(1, 2)`

**B** — `print("1" + "2")`

**C** — both of them

Set `q7_option` to `'A'`, `'B'` or `'C'`.

`print()` takes any number of values, converts each to text for you, and joins them with a
**space**. The `+` operator on two strings joins them with nothing at all — and on a string
and a number it does not work.

Two different jobs that look alike from a distance: `print` is for showing things to a
person, `+` is for building a string.

In [ ]:
# a scratch cell
print(1, 2)
print("1" + "2")

q7_option = ...

In [ ]:
grader.check("q7")

## Beyond the homework

Not graded:

- Put it together: write `receipt(names, prices)` that prints the whole block from the top
  of this notebook, header and dashed rule included, with the totals line at the bottom.
- Make the ticket names line up in a fixed-width column instead of relying on tabs.
  `f"{name:<10}"` left-aligns in 10 characters; `:>10` right-aligns. Which reads better for
  names, and which for prices?
- `to_cents('twelve fifty')` raises a `ValueError`. What *should* a ticket site do when a
  customer types that? Describe the behaviour before you write any code for it.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
